
## Training Your Own Linear Regressor

A linear regressor makes a prediction by computing the sum of the input features plus the bias term, in my code this is the intercept term, this is the hypothesis function:
\begin{equation*}
\begin{aligned}
h(\mathbf{x}^{(i)}; \mathbf{w}) &= y_{\text{pred}}^{(i)} \\
&= \mathbf{w}^{\top}\mathbf{x}^{(i)} \quad (\mathbf{x}^{(i)} \text{ with bias element}) \\
&= w_0 \cdot 1 + w_1 x_1^{(i)} + w_2 x_2^{(i)} + \dots + w_d x_d^{(i)}
\end{aligned}
\end{equation*}
Where d is the number of features. To determine the weights w, so that the hypothesis function actually is able predict y-values based on a arbitrary x-value, the weigths are tuned iteratively to minimize the MSE cost function for a Linear Regression model: 
\begin{equation*}
J=\text{MSE}(\mathbf{X}, \mathbf{y_{true};\mathbf{w}}) = \frac{1}{n} \sum_{i=1}^{n} \left(\mathbf{X}\mathbf{w} - \mathbf{y_{true}} \right)^{2} 
\end{equation*}
Which is proportional to the more well known RMSE, and to the loss function J.\\
From the MSE cost function, the closed form solution for the optimal weight can be found:
\begin{equation*}
\mathbf{w}^* = (\mathbf{X}^{\top} \mathbf{X})^{-1} \mathbf{X}^{\top} \mathbf{y}_{\text{true}}
\end{equation*}
But to invert a matrix can be very computationally complex often of order $O(n^3)$, therefore the gradient decent is used instead. The gradient decent finds the minimum of the MSE, by using the partial derivative of the MSE with respect the weights:
\begin{equation*}
\nabla_{\mathbf{w}} J = \left[ \frac{\partial J}{\partial w_1} \quad \frac{\partial J}{\partial w_2} \quad \dots \quad \frac{\partial J}{\partial w_d} \right]^{\top} = \frac{2}{n} \mathbf{X}^{\top} (\mathbf{Xw} - \mathbf{y}_{\text{true}})
\end{equation*}
This is a iteratively process where the next weight is found using:
\begin{equation*}
\mathbf{w}^{(step\, N+1)} = \mathbf{w}^{(step\, N)} - \eta \nabla_{\mathbf{w}} J
\end{equation*}
This can be implemented for the whole training set this is called batch gradient decent, this would be a good approach to find the minimum of the cost function, but can be slow for a large training set. Another method is picking a single random instance this is called stochastic gradient decent, it will be much faster for a large training set, but the cost function will bounce up and down.

In [2]:
import numpy as np

class MyLinReg():
    def __init__(self, eta0=0.03, max_iter=500, tol=1e-3, n_iter_no_change=5, verbose=True):
        self.eta0 = eta0
        self.max_iter = max_iter
        self.tol = tol
        self.n_iter_no_change = n_iter_no_change
        self.verbose = verbose
        self.coef_ = None
        self.intercept_ = None

    def _add_bias(self, X):
        # Prepend a column of 1s to handle the intercept (bias)
        return np.c_[np.ones((X.shape[0], 1)), X]

    def fit(self, X, y, method='batch'):
        """
        Fits the model using either 'batch' or 'sgd' (Stochastic Gradient Descent).
        """
        X_b = self._add_bias(X)
        n_samples, n_features = X_b.shape
        self.coef_ = np.zeros(n_features) # Initialize weights to zero
        
        for epoch in range(self.max_iter):
            if method == 'batch':
                # Batch Gradient Descent: Use all samples at once
                gradients = (2/n_samples) * X_b.T @ (X_b @ self.coef_ - y)
                self.coef_ = self.coef_ - self.eta0 * gradients
            
            elif method == 'sgd':
                # Stochastic Gradient Descent: Update per random sample
                indices = np.random.permutation(n_samples)
                for i in indices:
                    xi = X_b[i:i+1]
                    yi = y[i:i+1]
                    gradients = 2 * xi.T @ (xi @ self.coef_ - yi)
                    self.coef_ = self.coef_ - self.eta0 * gradients
            
            if self.verbose and epoch % 1 == 0:
                print(f"Epoch {epoch}: Loss {self.mse(X, y):.4f}")

    def predict(self, X):
        X_b = self._add_bias(X)
        return X_b @ self.coef_

    def mse(self, X, y_true):
        y_pred = self.predict(X)
        return np.mean((y_pred - y_true)**2)

    def score(self, X, y_true):
        """Calculates the R^2 score."""
        y_pred = self.predict(X)
        ss_res = np.sum((y_true - y_pred)**2)
        ss_tot = np.sum((y_true - np.mean(y_true))**2)
        return 1 - (ss_res / ss_tot)

    def __str__(self):
        return f"MyLinReg(eta0={self.eta0}, max_iter={self.max_iter})"


### Qf: Smoke testing

In [3]:
# Mini smoke test for your linear regressor: TestMyLinReg

import sys
import numpy

### SOME NIFTY HELPER FUNS ###

def isVector(y, expected_n=-1):
    assert isinstance(y, numpy.ndarray), f"expected type 'numpy.array' but got {type(y)}"
    assert y.ndim==1, f"expected y.ndim==1 but got {y.ndim}"
    assert expected_n<0 or expected_n==y.shape[0], f"expected vector of size {expected_n} but got size {y.shape}"
    return True

def isMatrix(X, expected_m=-1, expected_n=-1):
    assert isinstance(X, numpy.ndarray), f"expected type 'numpy.array' but got {type(X)}"
    assert X.ndim==2, f"expected X.ndim==2 but got {X.ndim}"
    assert expected_m<0 or expected_m==X.shape[0], f"expected matrix of size {expected_m}x{expected_n} but got size {X.shape}"
    assert expected_n<0 or expected_n==X.shape[1], f"expected vector of size {expected_m}x{expected_n} but got size {X.shape}"
    return True

def PrintMatrix(x, label="", precision=12, linewidth=60):
    hasFancy = False
    try:
        # NOTE: how does multiple import behave, any performance issues?
        from libitmal.utils import PrintMatrix as FancyPrintMatrix
        hasFancy = True
    except Exception as ex:
        Warn("could not import PrintMatrix from libitmal.utils, defaulting to simple function..")

    if hasFancy:
        FancyPrintMatrix(x, label=label, precision=precision, linewidth=linewidth)
    else:
        # default simple implementation
        print(f"{label}{' ' if len(label)>0 else ''}{x}")

def Col(color):
    hasFancy = False
    try:
        from libitmal.Utils.colors import Col as FancyCol
        hasFancy = True
    except Exception as ex:
        Warn("could not import Col from libitmal.Utils.colors, defaulting to simple function..")

    if hasFancy:
        return FancyCol(color)
    else:
        return ""

def ColEnd():
    hasFancy = False
    try:
        from libitmal.Utils.colors import ColEnd as FancyColEnd
        hasFancy = True
    except Exception as ex:
        Warn("could not import Col from libitmal.Utils.colors, defaulting to simple function..")

    if hasFancy:
        return FancyColEnd()
    else:
        return ""

def PrintOutput(msg, pre_msg, ex=None, color="", filestream=sys.stdout):

    def FormatTxt(txt, linewidth=60, prefix="", replacetabs=True):
        assert isinstance(txt, str)
        assert isinstance(linewidth, int) and linewidth > 0
        assert isinstance(prefix, str)

        if replacetabs:
            txt = txt.replace("\t","    ")

        r = ""
        n = 0
        m = 0
        for i in txt:
            m += 1
            if n >= linewidth:
                if not i.isspace() and m < len(txt) and not txt[m].isspace():
                    r += "\\" # add hypen
                r += "\n" + prefix
                n = 0

            if n == 0 and i.isspace():
                continue # skip leading space

            r += i
            n += 1

            if i == "\n":
                r += prefix
                n = 0

        return r

    col_beg = Col(color)
    col_end = ColEnd()

    prefix = "".ljust(len(pre_msg)) 
    msg = FormatTxt(msg, prefix=prefix)
    
    print(f"{col_beg}{pre_msg}{msg}{col_end}\n", file=filestream)

    if ex is not None:
        #msg += f"\n   EXCEPTION: {ex} ({type(ex)})"
        PrintOutput(str(ex), prefix + "EXCEPTION: ", None, "red", filestream)


def Warn(msg, ex=None):
    PrintOutput(msg, "WARN:  ", ex, "lyellow")


def Err(msg, ex=None):
    PrintOutput(msg, "ERROR: ", ex, "lred" )
    raise Exception(msg) if ex is None else ex


def Info(msg):
    PrintOutput(msg, "INFO:  ", None, "lpurple")


def SimpleAssertInRange(x, expected, eps):
    #assert isinstance(x, numpy.ndarray)
    #assert isinstance(expected, numpy.ndarray)
    #assert x.ndim==1 and expected.ndim==1
    #assert x.shape==expected.shape
    assert eps>0
    assert numpy.allclose(x, expected, eps) # should rtol or atol be set to eps?


def GenerateData():
    X = numpy.array([[8.34044009e-01],[1.44064899e+00],[2.28749635e-04],[6.04665145e-01]])
    y = numpy.array([5.97396028, 7.24897834, 4.86609388, 3.51245674])
    return X, y


def TestMyLinReg():
    X, y = GenerateData()

    try:
        # assume that your regressor class is named 'MyLinReg', please update/change
        regressor = MyLinReg()
    except Exception as ex:
        Err("your regressor has another name, than 'MyLinReg', please change the name in this smoke test", ex)

    try:
        regressor = MyLinReg(max_iter=200)
    except Exception as ex:
        Err("your regressor can not be constructed via the __init_ for parameter 'max_iter'", ex)
    try:
        regressor = MyLinReg(eta0=0.01)
    except Exception as ex:
        Err("your regressor can not be constructed via the __init_ for parameter 'eta0'", ex)
    try:
        regressor = MyLinReg(verbose=False)
    except Exception as ex:
        Warn("your regressor can not be constructed via the __init_ for parameter 'verbose'", ex)
    try:
        regressor = MyLinReg(tol=1e-3)
    except Exception as ex:
        Warn("your regressor can not be constructed via the __init_ for parameter 'tol'", ex)
    try:
        regressor = MyLinReg(n_iter_no_change=1e-3)
    except Exception as ex:
        Warn("your regressor can not be constructed via the __init_ for parameter 'n_iter_no_change'", ex)

    # create regressor with default hyperparameter values
    # to be used for training, prediction and score..
    try:
        regressor = MyLinReg()
    except Exception as ex:
        Err("your regressor can not be constructed via the __init_ with default parameters", ex)

    try:
        regressor.fit(X, y)
    except Exception as ex:
        Err("your regressor can not fit", ex)

    try:
        y_pred = regressor.predict(X)
        Info(f"y_pred = {y_pred}")
    except Exception as ex:
        Err("your regressor can not predict", ex)

    try:
        score  = regressor.score(X, y)
        Info(f"SCORE = {Col('lblue')}{score}{ColEnd()}")
    except Exception as ex:
        Err("your regressor fails in the score call", ex)


    try:
        w    = None # default
        bias = None # default
        try:
            w = regressor.coef_
            bias = regressor.intercept_
        except Exception as ex:
            w = None
            Warn("your regressor has no coef_/intercept_ atrributes, trying Weights() instead..", ex)
        try:
            if w is None:
                w = regressor.Weights() # maybe a Weigths function is avalible on you model?
                try:
                    assert w.ndim == 1,     "can only handle vector like w's for now"
                    assert w.shape[0] >= 2, "expected length of to be at least 2, that is one bias one coefficient"
                    bias = w[0]
                    w = w[1:]
                except Exception as ex:
                    w = None
                    Err("having a hard time concantenating our bias and coefficients, giving up!", ex)
        except Exception as ex:
            w = None
            Err("your regressor also has no Weights() function, giving up!", ex)
        Info(f"bias         = {bias}")
        Info(f"coefficients = {w}")
    except Exception as ex:
        Err("your regressor fails during extraction of bias and weights (but is a COULD)", ex)

    try:
        from libitmal.utils import PrintMatrix
    except Exception as ex:
        PrintMatrix = SimplePrintMatrix # fall-back
        Warn("could not import PrintMatrix from libitmal.utils, defaulting to simple function..")

    try:
        from libitmal.utils import AssertInRange
    except Exception as ex:
        AssertInRange = SimpleAssertInRange # fall-back
        Warn("could not import AssertInRange from libitmal.utils, defaulting to simple function..")

    try:
        if w is not None:
            if bias is not None:
                w = numpy.concatenate(([bias], w)) # re-concat bias an coefficients, may be incorrect for your implementation!
            
            # TEST VECTOR:
            w_expected = numpy.array([4.046879011698, 1.880121487278])
            
            PrintMatrix(w,          label="       w         =")
            PrintMatrix(w_expected, label="       w_expected=")
            print()
            
            eps = 1E-2 # somewhat big epsilon, allowing some slack..
            AssertInRange(w, w_expected, eps)
            Info("Well, good news, your w and the expected w-vector seem to be very close numerically, so the smoke-test has passed!")
            
            return regressor
        else:
            Warn("cannot test due to missing w information")
    except Exception as ex:
        Err("mini-smoketest on your regressor failed", ex)
    
    return None

Warn("This mini smoke-test may produce false-positives and/or\n false-negatives..")
TestMyLinReg()

print("OK")

WARN:  This mini smoke-test may produce false-positives and/or
       false-negatives..

Epoch 0: Loss 25.5353
Epoch 1: Loss 21.0291
Epoch 2: Loss 17.3557
Epoch 3: Loss 14.3609
Epoch 4: Loss 11.9192
Epoch 5: Loss 9.9284
Epoch 6: Loss 8.3052
Epoch 7: Loss 6.9814
Epoch 8: Loss 5.9018
Epoch 9: Loss 5.0212
Epoch 10: Loss 4.3028
Epoch 11: Loss 3.7167
Epoch 12: Loss 3.2383
Epoch 13: Loss 2.8478
Epoch 14: Loss 2.5289
Epoch 15: Loss 2.2684
Epoch 16: Loss 2.0554
Epoch 17: Loss 1.8813
Epoch 18: Loss 1.7389
Epoch 19: Loss 1.6222
Epoch 20: Loss 1.5266
Epoch 21: Loss 1.4481
Epoch 22: Loss 1.3836
Epoch 23: Loss 1.3305
Epoch 24: Loss 1.2868
Epoch 25: Loss 1.2506
Epoch 26: Loss 1.2207
Epoch 27: Loss 1.1958
Epoch 28: Loss 1.1751
Epoch 29: Loss 1.1577
Epoch 30: Loss 1.1432
Epoch 31: Loss 1.1308
Epoch 32: Loss 1.1204
Epoch 33: Loss 1.1114
Epoch 34: Loss 1.1037
Epoch 35: Loss 1.0970
Epoch 36: Loss 1.0912
Epoch 37: Loss 1.0860
Epoch 38: Loss 1.0815
Epoch 39: Loss 1.0774
Epoch 40: Loss 1.0737
Epoch 41: Loss

### Qh Conclusion

It does take some time to fine-tune the regressor with the correct eto0 and max_iter, but the model will converge to the correct weigths when using the slow and steady batch gradient decent model, but the stochastic gradient decent model is not stable, to find a use-case for the SGD an optimization of the model would be to first use the SGD to find an approximate of the weigths and afterwards use the batch model. 

REVISIONS||
:- | :- |
2022-12-22| CEF, initial draft. 
2023-02-26| CEF, first release.
2023-02-28| CEF, fix a few issues related to import from libitmal, added Info and color output.
2024-09-19| CEF, major overhaul, change math/text and code snippets.
2024-09-25| CEF, final fixes, tests, and proof-reading. Moved early stopping and learning graphs to a later excercise.
2024-10-04| CEF, clarified Qa with respect to what-is-to-be implemented and what-is-to-be described in text only.
2025-03-14| CEF, fixed error in isMatrix, use X.shape instead of y.shape.
2025-09-25| CEF, removed TODO comments in this table.
2026-02-09| CEF, fixed image links.